<a href="https://colab.research.google.com/github/karkessler/dhbw-mathe3/blob/main/notebooks/numerik/schwingungsdifferentialgleichung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Schwingungsdifferentialgleichung

Wir untersuchen die freie Schwingung eines Feder-Masse-Systems analytisch und numerisch. Anschließend ergänzen wir eine lineare Dämpfung. Das Notebook benötigt nur **NumPy** und **Matplotlib**.

![Feder-Masse-System mit Auslenkung und Rückstellkraft](assets/schwingungsdifferentialgleichung.png)

Bei positiver Auslenkung $x(t)$ wirkt die Rückstellkraft $F=-kx$ zurück zur Ruhelage.


## Aufgabe

Ein Körper der Masse $m=0{,}5\,\mathrm{kg}$ ist an einer Feder mit der Federkonstanten $k=8\,\mathrm{N/m}$ befestigt. Reibung wird zunächst vernachlässigt. Es gilt

$$x(0)=0{,}10\,\mathrm m,\qquad x'(0)=0.$$

1. Stelle aus $m x''=-kx$ die Schwingungsdifferentialgleichung auf.
2. Löse das Anfangswertproblem und bestimme Kreisfrequenz und Periodendauer.
3. Formuliere die Differentialgleichung als System erster Ordnung. Berechne auf $0\le t\le5$ eine RK4-Lösung mit $h=0{,}02$ und vergleiche sie mit der analytischen Lösung.
4. Ergänze eine Dämpfung $c=0{,}8\,\mathrm{Ns/m}$ und beschreibe die Veränderung.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m = 0.5       # kg
k = 8.0       # N/m
x0 = 0.10     # m
v0 = 0.0      # m/s
h = 0.02      # s
t_end = 5.0   # s


## Lösung 1 und 2: analytische Lösung

Aus $m x''=-kx$ folgt

$$x''+\frac{k}{m}x=0,\qquad x''+16x=0.$$

Mit $\omega_0=\sqrt{k/m}=4\,\mathrm{s}^{-1}$ lautet die allgemeine Lösung

$$x(t)=C_1\cos(4t)+C_2\sin(4t).$$

Die Anfangsbedingungen ergeben $C_1=0{,}10$ und $C_2=0$. Also

$$x(t)=0{,}10\cos(4t)\,\mathrm m,\qquad T=\frac{2\pi}{\omega_0}=\frac{\pi}{2}\,\mathrm s.$$


In [ ]:
omega0 = np.sqrt(k / m)
T = 2 * np.pi / omega0

def x_exakt(t):
    return x0 * np.cos(omega0 * t) + (v0 / omega0) * np.sin(omega0 * t)

print(f"Kreisfrequenz omega_0 = {omega0:.3f} 1/s")
print(f"Periodendauer T          = {T:.6f} s")


## Lösung 3: System erster Ordnung und RK4

Mit $v=x'$ wird die Gleichung zum System

$$\begin{pmatrix}x\\v\end{pmatrix}'=\begin{pmatrix}v\\-(k/m)x\end{pmatrix}.$$


In [ ]:
def rechte_seite(t, zustand, daempfung=0.0):
    x, v = zustand
    return np.array([v, -(daempfung / m) * v - (k / m) * x])

def rk4(f, y0, t0, t_end, h):
    n = int(round((t_end - t0) / h))
    t = np.linspace(t0, t_end, n + 1)
    y = np.empty((n + 1, len(y0)), dtype=float)
    y[0] = y0
    for i in range(n):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h/2, y[i] + h*k1/2)
        k3 = f(t[i] + h/2, y[i] + h*k2/2)
        k4 = f(t[i] + h, y[i] + h*k3)
        y[i + 1] = y[i] + h * (k1 + 2*k2 + 2*k3 + k4) / 6
    return t, y

t, y = rk4(rechte_seite, np.array([x0, v0]), 0.0, t_end, h)
fehler = np.abs(y[:, 0] - x_exakt(t))
print(f"Maximaler Fehler: {fehler.max():.3e} m")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(t, x_exakt(t), label="analytisch", linewidth=2.5)
ax.plot(t, y[:, 0], "--", label=f"RK4, h={h}")
ax.set(xlabel="Zeit t [s]", ylabel="Auslenkung x [m]",
       title="Ungedämpfte harmonische Schwingung")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## Lösung 4: gedämpfte Schwingung

Für $c=0{,}8\,\mathrm{Ns/m}$ gilt

$$m x''+cx'+kx=0,\qquad x''+1{,}6x'+16x=0.$$

Da $c^2-4mk<0$, ist das System unterkritisch gedämpft. Es schwingt weiter, aber die Amplitude nimmt mit der Hüllkurve $e^{-ct/(2m)}=e^{-0{,}8t}$ ab.


In [ ]:
c = 0.8  # Ns/m
t_d, y_d = rk4(lambda t, z: rechte_seite(t, z, daempfung=c),
               np.array([x0, v0]), 0.0, t_end, h)
omega_d = np.sqrt(k/m - (c/(2*m))**2)
huellkurve = x0 * np.exp(-c * t_d / (2*m))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(t, y[:, 0], label="ungedämpft", alpha=0.75)
ax.plot(t_d, y_d[:, 0], label="gedämpft", linewidth=2.5)
ax.plot(t_d, huellkurve, "k:", label="Hüllkurve")
ax.plot(t_d, -huellkurve, "k:")
ax.set(xlabel="Zeit t [s]", ylabel="Auslenkung x [m]",
       title="Einfluss der linearen Dämpfung")
ax.grid(alpha=0.3)
ax.legend()
plt.show()
print(f"Gedämpfte Kreisfrequenz omega_d = {omega_d:.3f} 1/s")


## Eigene Experimente

Ändere oben $m$, $k$, $x_0$, $v_0$, $h$ oder $c$ und führe die Zellen erneut aus. Beobachte insbesondere:

- Wie verändert sich die Periodendauer bei größerer Masse?
- Wie wirkt eine steifere Feder?
- Ab welcher Dämpfung verschwindet die Schwingung?
- Wie verändert die Schrittweite $h$ den RK4-Fehler?
